#### New Note to generate the Tables and Images comparing XGBoost & BOOMER.

#### 250929: Hao Hu

In [3]:
import numpy as np
import json
import pandas as pd
import os
import builtins
from collections import Counter

In [ ]:
# Step 1: Transform the results of JSON format into the dataframe

# read XGBoost and BOOMER reults in JSON format
df_xgboost = pd.read_json('./result_xgboost_251022.json')
df_boomers = pd.read_json('./result_boomers_251022.json')

df_xgboost['method'] = 'xgboost'
df_boomers['method'] = 'boomer'

datasets = sorted(set(df_xgboost['dataset']))

# concat the two results dataframes into one
df_xgboost_boomer = pd.concat([df_xgboost, df_boomers])

df_xgboost_boomer.info()


<class 'pandas.core.frame.DataFrame'>
Index: 1260 entries, 0 to 629
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   dataset             1260 non-null   object 
 1   n_estimator         1260 non-null   int64  
 2   max_depth           1260 non-null   int64  
 3   train_acc           1260 non-null   float64
 4   test_acc            1260 non-null   float64
 5   num_feat            1260 non-null   int64  
 6   num_class           1260 non-null   int64  
 7   num_instance        1260 non-null   int64  
 8   num_identical_path  1260 non-null   int64  
 9   method              1260 non-null   object 
dtypes: float64(2), int64(6), object(2)
memory usage: 108.3+ KB


In [1]:
## Functions preparing the Tables
def get_df_infos(df, method, max_depth, n_estimator, dataset, wdigit=None):
    assert (method in set(df['method'])) and (max_depth in set(df['max_depth'])) \
          and (n_estimator in set(df['n_estimator']) and dataset in set(df['dataset']))
    
    if not wdigit is None:
        assert (wdigit in set(df['wdigit']))
        df_infos = df[(df['method'] == method) & (df['max_depth'] == max_depth) \
                & (df['n_estimator'] == n_estimator) & (df['dataset'] == dataset) & (df['wdigit'] == wdigit)] 
    else:
        df_infos = df[(df['method'] == method) & (df['max_depth'] == max_depth) \
                & (df['n_estimator'] == n_estimator) & (df['dataset'] == dataset)]
    assert df_infos.shape[0] == 1
    return df_infos

def value2str(v):
    if isinstance(v, np.float64):
        return f'{v:.3f}'
    elif isinstance(v, np.int64):
        return str(v)
    else:
        return v

def str_with_bold_value(df1, df2, col_names, maxs):  
    assert len(col_names) == len(maxs)
    better_infos = []
    str_row = ''
    # 251002: add the value #wins (if equal, both are #wins + 1)
    wins_d1, wins_d2 = {}, {}
    for i in range(len(col_names)):
        wins_d1[col_names[i]] = 0
        wins_d2[col_names[i]] = 0
        if df1.iloc[-1][col_names[i]] == df2.iloc[-1][col_names[i]]:
            better_infos.append(False)
            str_row += '\\textbf{' + value2str(df1.iloc[-1][col_names[i]]) + '} & ' 
            wins_d1[col_names[i]] += 1
            wins_d2[col_names[i]] += 1
        elif (df1.iloc[-1][col_names[i]] > df2.iloc[-1][col_names[i]]) ^ maxs[i]:
            better_infos.append(False)
            str_row += value2str(df1.iloc[-1][col_names[i]]) + ' & '
            wins_d2[col_names[i]] += 1
        else:
            better_infos.append(True)
            str_row += '\\textbf{' + value2str(df1.iloc[-1][col_names[i]]) + '} & '
            wins_d1[col_names[i]] += 1
    for i in range(len(col_names) - 1):
        str_row += value2str(df2.iloc[-1][col_names[i]]) + ' & ' if better_infos[i] else '\\textbf{' + value2str(df2.iloc[-1][col_names[i]]) + '} & '
    str_row += value2str(df2.iloc[-1][col_names[-1]]) + ' \\\\' if better_infos[-1] else '\\textbf{' + value2str(df2.iloc[-1][col_names[-1]]) + '} \\\\'
    return str_row, wins_d1, wins_d2

def deal_wins(win_all, win_specific):
    if win_all is None:
        win_all = Counter(win_specific)
    else:
        win_all = win_all + Counter(win_specific)
    return win_all

def wins2str(wins, keys_given):
    wins2str = ''
    for i in range(len(keys_given) - 1):
        wins2str += str(wins[keys_given[i]]) + ' & '
    wins2str += str(wins[keys_given[-1]]) + ' '
    return wins2str

def prepare_table_by_bench_separ(df, out_base_path='latex_separ'):
    if not os.path.exists(out_base_path):
        os.mkdir(out_base_path)
    datasets_infos = sorted(set(df['dataset']))
    cols_selected = ['train_acc', 'test_acc', 'num_identical_path']

    for d_info in datasets_infos:
        out_path = os.path.join(out_base_path, d_info.replace('\n', '') + '_separ.tex')
        win_b_all, win_x_all = None, None
        # 1st col (m_depth) => 2nd col (num_estimator) => Boomer(train/test/num_identical_path), XGBoost
        num_estimators = sorted(set(df['n_estimator']))
        with builtins.open(out_path, 'w') as f:
            for m_depth in sorted(set(df['max_depth'])):
                # prepare each multirow cell
                f.write('\\multirow{' + str(len(num_estimators)) + '}{*}{' + str(m_depth) + '}\n')
                for n in num_estimators:
                    f.write('& ' + str(n) + ' & ')
                    df_infos_boomer = get_df_infos(df, 'boomer', m_depth, n, d_info)
                    df_infos_xgboost = get_df_infos(df, 'xgboost', m_depth, n, d_info)
                    str_row, win_boomer, win_xgboost = str_with_bold_value(df_infos_boomer, df_infos_xgboost, cols_selected, [True, True, False])
                    f.write(str_row + '\n')
                    win_b_all = deal_wins(win_b_all, win_boomer)
                    win_x_all = deal_wins(win_x_all, win_xgboost)
                f.write('\\hline \n')
            # dealing the last line of #WINS
            f.write('\\#W & / & ' + wins2str(win_b_all, cols_selected) + ' & ' + wins2str(win_x_all, cols_selected) + '\\\\ \\hline')

def prepare_table_by_bench_specific(df, out_bpath, num_estimator=50):
    if not os.path.exists(out_bpath):
        os.mkdir(out_bpath)
    out_path = os.path.join(out_bpath, '0_results_boomer_xgboost_base(' + str(num_estimator) + ').tex')
    datasets_infos = sorted(set(df['dataset']))
    mdepths = sorted(set(df['max_depth']))
    win_b_all, win_x_all = None, None
    cols_selected = ['train_acc', 'test_acc', 'num_identical_path']

    with builtins.open(out_path, 'w') as f:
        # 1st col (datasets) => 2nd col (max_depths) => Boomer (train/test/num_identical_path), XGBoost
        for d_info in datasets_infos:
            f.write('\\multirow{' + str(len(mdepths)) + '}{*}{\\shortstack[1]{' + d_info + '}}\n')
            for m_dpeth in mdepths:
                f.write('& ' + str(m_dpeth) + ' & ')
                df_infos_boomer = get_df_infos(df, 'boomer', m_dpeth, num_estimator, d_info)
                df_infos_xgboost = get_df_infos(df, 'xgboost', m_dpeth, num_estimator, d_info)
                str_row, win_boomer, win_xgboost = str_with_bold_value(df_infos_boomer, df_infos_xgboost, cols_selected, [True, True, False])
                win_b_all = deal_wins(win_b_all, win_boomer)
                win_x_all = deal_wins(win_x_all, win_xgboost)
                f.write(str_row + '\n')
            f.write('\\hline \n')
        # dealing the last line of #WINS
        f.write('\\#WINS & / & ' + wins2str(win_b_all, cols_selected) + ' & ' + wins2str(win_x_all, cols_selected) + '\\\\ \\hline')

def prepare_table_by_bench(df, out_bpath, num_estimator=50, table_separ=False):

    if table_separ:
        prepare_table_by_bench_separ(df, out_bpath)
    else:
        prepare_table_by_bench_specific(df, out_bpath, num_estimator)


In [23]:
# Step 2: prepare the Tables to show the results

# Table 1: A representative case (num_estimator=50, max_depth=[4,5,6], all_benchs)
# df_n_estimator_50 = df_xgboost_boomer.loc[df_xgboost_boomer['n_estimator'] == 50]
# renames the cells of the column 'dataset' with the dataset infos
add_info_format1 = lambda r: r['dataset'] + '(' + str(r['num_feat']) + ' - ' + str(r['num_class']) + ' - ' + str(r['num_instance']) + ')'
df_xgboost_boomer_c = df_xgboost_boomer.copy()
df_xgboost_boomer_c['dataset'] = df_xgboost_boomer_c.apply(add_info_format1, axis='columns')
prepare_table_by_bench(df_xgboost_boomer_c, 'tex_table', table_separ=False, num_estimator=50)


# Table 2s: Separates tables for each benchmarks.
prepare_table_by_bench(df_xgboost_boomer_c, 'tex_table', table_separ=True)

In [16]:
# 251125: prepare the dataframe to show the Boomers with different weight digits
df_boomers_wd = pd.read_json('./result_boomers_251125.json')
df_boomers_wd = df_boomers_wd.where(pd.notnull(df_boomers_wd), -1)
df_boomers_wd['method'] = 'boomer'

def table_boomer_diff_wd(df, out_base_path='latex_separ'):
    '''
        251125: prepare the table to show the prediction quality with different wd
    '''
    if not os.path.exists(out_base_path):
        os.mkdir(out_base_path)
    datasets_infos = sorted(set(df['dataset']))
    cols_selected = ['train_acc', 'test_acc']

    for d_info in datasets_infos:
        out_path = os.path.join(out_base_path, d_info.replace('\n', '') + '_separ.tex')
        # 1st col (m_depth) => 2nd col (num_estimator) => Boomer (train/test) for each w_digit
        num_estimators = sorted(set(df['n_estimator']))
        wdigits = sorted(set(df['wdigit']))
        # print(wdigits)

        with builtins.open(out_path, 'w') as f:
            for m_depth in sorted(set(df['max_depth'])):
                # prepare each multirow cell
                f.write('\\multirow{' + str(len(num_estimators)) + '}{*}{' + str(m_depth) + '}\n')
                for n in num_estimators:
                    f.write('& ' + str(n) + ' & ')
                    str_row = ''
                    df_orig_infos = get_df_infos(df, 'boomer', m_depth, n, d_info, wdigit=-1)
                    for wd in wdigits:
                        df_infos = get_df_infos(df, 'boomer', m_depth, n, d_info, wdigit=wd)
                        for col in cols_selected:
                            # presenting the differenc to the orig instead of value
                            if wd != -1.0:
                                str_row += value2str(df_infos.iloc[-1][col] - df_orig_infos.iloc[-1][col]) + ' & '
                            else:
                                str_row += value2str(df_infos.iloc[-1][col]) + ' & '
                    f.write(str_row[:-2] + '\\\\\n')
                f.write('\\hline \n')

table_boomer_diff_wd(df_boomers_wd)

### Preapring the images!!

In [24]:
# renames the cells of the column 'dataset' with the dataset infos
# add_info_format2 = lambda r: r['dataset'] + '\n(F=' + str(r['num_feat']) + '_C=' + str(r['num_class']) + '_I=' + str(r['num_instance']) + ')'
add_info_format1 = lambda r: r['dataset'] + '(' + str(r['num_feat']) + '|' + str(r['num_class']) + '|' + str(r['num_instance']) + ')'
df_xgboost_boomer['dataset'] = df_xgboost_boomer.apply(add_info_format1, axis='columns')

# prepare the dataframe with selected num_estimator
n_estimators = [10, 30, 50, 70, 90]
df_xgboost_boomer_subset = df_xgboost_boomer[df_xgboost_boomer['n_estimator'].isin(n_estimators)]
df_xgboost_boomer_subset['method-num'] = df_xgboost_boomer_subset['method'] + '-' + df_xgboost_boomer_subset['n_estimator'].astype(str)
# print(df_xgboost_boomer_subset)


/var/folders/dq/71xl15nn24s1rm05xjrkfnlm0000gn/T/ipykernel_64013/4116551659.py:9: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [25]:
import plotly.express as px
import plotly.graph_objects as pgo
import plotly.subplots as ps

# Step 3: prepare the radar charts to show the reslts
# theta => datasets with infos; color => num_estimator [10, 30, 50, 70, 90]; methods => ['boomer', 'xgboost']
# Radar Chart 1s: radius = train_acc; different charts with different max_depth

# radar1_figs = []
mdepths = sorted(set(df_xgboost_boomer_subset['max_depth']))
# preparing the set of increasing colors of 2 groups (blue / red)
blue_increase = ["#8AC4FF","#00FFFF","#0080FF",  "#0000FF", "#000075"]
red_increase = ["#FFB8B8","#FFAD5C","#FF8000","#FF0000","#A30000"]
colors_two_group = blue_increase[0:len(n_estimators)] + red_increase[0:len(n_estimators)]
line_dash_maps = {'xgboost-10': 'dot', 'xgboost-30': 'dot', 'xgboost-50': 'dot', 'xgboost-70': 'dot', 'xgboost-90': 'dot', 
               'boomer-10': 'solid', 'boomer-30': 'solid', 'boomer-50': 'solid', 'boomer-70': 'solid', 'boomer-90': 'solid'}

for i in range(len(mdepths)):
    df_results_mdepth = df_xgboost_boomer_subset[df_xgboost_boomer_subset['max_depth'] == mdepths[i]]
    fig = px.line_polar(df_results_mdepth, r='train_acc', theta='dataset',
                    color='method-num', line_close=True, 
                    line_dash_map=line_dash_maps, line_dash='method-num',
                    width=800, height=600,
                    color_discrete_sequence=colors_two_group,
                    template='plotly_white', range_r=[0.8, 1.01], 
                    title='Train_acc (max_depth/max_cond = ' + str(mdepths[i]) + ')')
    fig.update_polars(radialaxis_tickvals=[0.8, 0.9, 1.0], radialaxis_ticks="inside",
                      angularaxis_tickfont_size=10, angularaxis_tickfont_lineposition='under')
    fig.update_layout(title_xanchor='center', title_x=0.5, 
                    legend_title='method-num', legend_x=-0.2, legend_y=0.5, legend_font_size=10)
                    # , legend_xanchor='right')
    fig.show()

In [26]:
# Chart 2: radius = test_acc; different charts with different max_depth
for i in range(len(mdepths)):
    df_results_mdepth = df_xgboost_boomer_subset[df_xgboost_boomer_subset['max_depth'] == mdepths[i]]
    fig = px.line_polar(df_results_mdepth, r='test_acc', theta='dataset',
                    color='method-num', line_close=True, 
                    line_dash_map=line_dash_maps, line_dash='method-num',
                    color_discrete_sequence=colors_two_group,
                    width=800, height=600,
                    template='plotly_white', range_r=[0.65, 1.01], 
                    title='Test_acc (max_depth/max_cond = ' + str(mdepths[i]) + ')')
    fig.update_polars(radialaxis_tickvals=[0.65, 0.8, 1.0], radialaxis_ticks="inside",
                      angularaxis_tickfont_size=10, angularaxis_tickfont_lineposition='under')
    fig.update_layout(title_xanchor='center', title_x=0.5, 
                      # showlegend=False)
                    legend_title='method-num',legend_x=-0.2, legend_y=0.5, legend_font_size=10)
    fig.show()


In [27]:
# Chart 3: radius = num_identical_path; different charts with different max_depth
for i in range(len(mdepths)):
    df_results_mdepth = df_xgboost_boomer_subset[df_xgboost_boomer_subset['max_depth'] == mdepths[i]]
    fig = px.line_polar(df_results_mdepth, r='num_identical_path', theta='dataset',
                    color='method-num', line_close=True, 
                    width=800, height=600,
                    line_dash_map=line_dash_maps, line_dash='method-num',
                    color_discrete_sequence=colors_two_group, log_r=True,
                    template='plotly_white', range_r=[10, 3000], 
                    title='Number of identical paths (max_depth/max_cond = ' + str(mdepths[i]) + ')')
    fig.update_polars(radialaxis_tickvals=[10, 100, 1000, 3000], radialaxis_ticks="inside",
                      angularaxis_tickfont_size=10, angularaxis_tickfont_lineposition='under')
    fig.update_layout(title_xanchor='center', title_x=0.5, 
                      # showlegend=False)
                    legend_title='method-num', legend_x=-0.2, legend_y=0.5, legend_font_size=10)
    fig.show()